# 🔴 Day 12 — Subgraphs & Multi-Agent Systems

**LangGraph from Zero to Advanced · Day 12 of 16**

**Today's goal:** build big systems from small graphs, and make **several agents** work as a team.

| Lesson | Topic |
|---|---|
| 1 | Subgraph = a graph used as a node |
| 2 | Subgraph with its own private state |
| 3 | See inside subgraphs while streaming |
| 4 | Multi-agent: the **supervisor** pattern |
| 5 | Multi-agent: **handoffs** with `Command` |
| 6 | Practice |

In [ ]:
%pip install -q -U langgraph langchain langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

def show_graph(app, xray=False):
    """Draw the graph as a picture, or print it as text when offline. xray=True shows subgraph insides."""
    try:
        display(Image(app.get_graph(xray=xray).draw_mermaid_png()))
    except Exception:
        print(app.get_graph(xray=xray).draw_mermaid())

print("✅ Ready")

## Lesson 1 — Subgraph = a graph used as a node

A **compiled graph can be added as a node** in another graph. Like functions calling functions — build, test, and reuse pieces separately.

If the subgraph and the parent **share state keys**, just pass the compiled subgraph to `add_node`.

```
Parent:  START ► get_input ► [ clean_subgraph ] ► report ► END
Sub:                 START ► strip ► lower ► END
```

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class TextState(TypedDict):
    text: str
    report: str

# --- the subgraph ---
def strip_spaces(state): return {"text": " ".join(state["text"].split())}
def lower(state):        return {"text": state["text"].lower()}

sub = StateGraph(TextState)
sub.add_sequence([strip_spaces, lower])
sub.add_edge(START, "strip_spaces")
sub.add_edge("lower", END)
clean_subgraph = sub.compile()

# --- the parent graph ---
def get_input(state): return {"text": state["text"] + "   !!!"}
def report(state):    return {"report": f"cleaned → '{state['text']}'"}

parent = StateGraph(TextState)
parent.add_node("get_input", get_input)
parent.add_node("clean", clean_subgraph)        # 👈 a whole graph as one node
parent.add_node("report", report)
parent.add_edge(START, "get_input")
parent.add_edge("get_input", "clean")
parent.add_edge("clean", "report")
parent.add_edge("report", END)
parent_app = parent.compile()
show_graph(parent_app, xray=True)

print(parent_app.invoke({"text": "   HELLO    LangGraph  "})["report"])

## Lesson 2 — Subgraph with its own private state

Often the subgraph needs **different** keys. Then call it **inside a normal node** and translate:

1. parent state → subgraph input
2. `subgraph.invoke(...)`
3. subgraph output → parent update

Example: a reusable **"translate & check"** subgraph used by a parent that only knows `question` / `answer`.

In [ ]:
class TranslateState(TypedDict):     # subgraph's own keys
    source: str
    target_lang: str
    translated: str
    ok: bool

def translate(state):
    t = llm.invoke(f"Translate to {state['target_lang']}. Reply with only the translation: {state['source']}").content
    return {"translated": t}

def check(state):
    return {"ok": len(state["translated"].strip()) > 0}

t = StateGraph(TranslateState)
t.add_sequence([translate, check])
t.add_edge(START, "translate")
t.add_edge("check", END)
translator = t.compile()

class QAState(TypedDict):            # parent's keys — completely different
    question: str
    answer: str
    answer_hindi: str

def answer(state):
    return {"answer": llm.invoke(f"Answer in one sentence: {state['question']}").content}

def to_hindi(state):
    out = translator.invoke({"source": state["answer"], "target_lang": "Hindi"})   # 1 + 2
    return {"answer_hindi": out["translated"]}                                     # 3

p = StateGraph(QAState)
p.add_sequence([answer, to_hindi])
p.add_edge(START, "answer")
p.add_edge("to_hindi", END)
qa = p.compile()

res = qa.invoke({"question": "What is the capital of Maharashtra?"})
print("EN:", res["answer"])
print("HI:", res["answer_hindi"])

## Lesson 3 — See inside subgraphs while streaming

By default, streaming shows the subgraph as a single step. Add `subgraphs=True` to see the steps **inside** it. Each item becomes `(namespace, chunk)`.

In [ ]:
for namespace, chunk in parent_app.stream({"text": "  ABC  DEF "}, stream_mode="updates", subgraphs=True):
    where = "parent" if not namespace else f"inside {namespace[0].split(':')[0]}"
    print(f"{where:15} → {chunk}")

## Lesson 4 — Multi-agent: the supervisor pattern

One agent with 20 tools gets confused. Better: **specialist agents**, each with a few tools, and a **supervisor** that decides who works next.

```
                ┌──────► researcher ──┐
START ► supervisor ◄──────────────────┤
                └──────► writer ──────┘
                │
                └─ FINISH ► END
```

* **researcher** — an agent with a `search_notes` tool (a tiny fake knowledge base).
* **writer** — writes the final answer from the research.
* **supervisor** — LLM with structured output: `next = researcher | writer | FINISH`.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, SystemMessage
from langchain.agents import create_agent
from langgraph.graph import MessagesState
from langgraph.types import Command

KNOWLEDGE = {
    "langgraph": "LangGraph is a library for building stateful, multi-step LLM agents as graphs. Released by LangChain in 2024.",
    "checkpointer": "A checkpointer saves graph state after each step, enabling memory, human-in-the-loop and time travel.",
    "send": "Send lets a graph run a node many times in parallel with different inputs (map-reduce).",
}

@tool
def search_notes(query: str) -> str:
    """Search the internal notes. Use simple keywords like 'langgraph' or 'checkpointer'."""
    hits = [v for k, v in KNOWLEDGE.items() if k in query.lower()]
    return "\n".join(hits) if hits else "No notes found."

research_agent = create_agent(model=llm, tools=[search_notes],
                              system_prompt="You are a researcher. Search the notes and report the facts you found. Be brief.")

class TeamState(MessagesState):
    steps: int

def researcher(state: TeamState) -> Command[Literal["supervisor"]]:
    result = research_agent.invoke({"messages": state["messages"]})
    note = AIMessage(content=result["messages"][-1].content, name="researcher")
    return Command(update={"messages": [note]}, goto="supervisor")

def writer(state: TeamState) -> Command[Literal["supervisor"]]:
    reply = llm.invoke([SystemMessage(content="You are a writer. Using the research in the conversation, "
                                              "write a clear, friendly answer for a beginner in max 80 words.")]
                       + state["messages"])
    return Command(update={"messages": [AIMessage(content=reply.content, name="writer")]}, goto="supervisor")

class Next(BaseModel):
    next: Literal["researcher", "writer", "FINISH"] = Field(
        description="researcher if facts are missing, writer if facts exist but no final answer, FINISH if the writer has answered")

supervisor_llm = llm.with_structured_output(Next)

def supervisor(state: TeamState) -> Command[Literal["researcher", "writer", "__end__"]]:
    steps = state.get("steps", 0)
    if steps >= 4:                                   # safety limit
        return Command(goto=END)
    history = "\n".join(f"{getattr(m, 'name', None) or m.type}: {m.content[:200]}" for m in state["messages"])
    decision = supervisor_llm.invoke(f"You manage a researcher and a writer. Conversation so far:\n{history}\nWho should act next?")
    print(f"🧭 supervisor → {decision.next}")
    goto = END if decision.next == "FINISH" else decision.next
    return Command(update={"steps": steps + 1}, goto=goto)

g = StateGraph(TeamState)
g.add_node("supervisor", supervisor)
g.add_node("researcher", researcher)
g.add_node("writer", writer)
g.add_edge(START, "supervisor")
team = g.compile()
show_graph(team)

In [ ]:
result = team.invoke({"messages": [("user", "Explain what a checkpointer is in LangGraph.")], "steps": 0})
print()
for m in result["messages"]:
    print(f"[{getattr(m, 'name', None) or m.type}] {m.content}\n")

## Lesson 5 — Handoffs with `Command`

No supervisor: agents **hand off** directly to each other — like a call-centre transfer. Each agent returns `Command(goto="other_agent")` when the question isn't theirs.

```
START ► sales ⇄ support ► END
```

In [ ]:
class AgentReply(BaseModel):
    reply: str = Field(description="your answer, or empty if handing off")
    handoff: Literal["none", "sales", "support"] = Field(description="who should handle it, 'none' if you answered")

class DeskState(MessagesState):
    handoffs: int

def make_agent(name: str, role: str):
    brain = llm.with_structured_output(AgentReply)
    def agent(state: DeskState) -> Command[Literal["sales", "support", "__end__"]]:
        already_transferred = state.get("handoffs", 0) >= 1
        rule = ("The chat was already transferred to you, so you MUST answer (handoff='none')."
                if already_transferred else
                "If the question is not about your area, hand off instead of answering.")
        out = brain.invoke([SystemMessage(content=f"You are the {name} agent. {role} {rule}")] + state["messages"])
        if out.handoff not in ("none", name) and not already_transferred:
            print(f"🔀 {name} → hands off to {out.handoff}")
            return Command(update={"handoffs": 1}, goto=out.handoff)
        return Command(update={"messages": [AIMessage(content=out.reply, name=name)]}, goto=END)
    return agent

g = StateGraph(DeskState)
g.add_node("sales", make_agent("sales", "You handle prices, plans and discounts."))
g.add_node("support", make_agent("support", "You handle bugs, login problems and technical help."))
g.add_edge(START, "sales")                       # every chat starts at sales
desk = g.compile()
show_graph(desk)

for q in ["How much is the premium plan?", "I can't log in, it shows error 403"]:
    out = desk.invoke({"messages": [("user", q)], "handoffs": 0})
    print(f"👤 {q}\n🤖 [{out['messages'][-1].name}] {out['messages'][-1].content}\n")

| Pattern | Who decides the next agent | Good for |
|---|---|---|
| **Supervisor** | one central LLM | clear control, easy to debug |
| **Handoffs** | each agent itself | customer-service style transfers |
| **Hierarchical** | supervisors of supervisors (subgraphs!) | very large teams |

## Lesson 6 — Practice

**Exercise 1.** Make a subgraph `score_text` (keys: `text`, `score`) that sets `score` = number of words. Use it as a node in a parent that has the same keys plus `label` (`"long"` if score > 5).

**Exercise 2.** Add a third specialist **`fact_checker`** to the supervisor team (an LLM node that says whether the writer's answer matches the research). Update the `Next` model and the graph.

In [ ]:
# ✅ Solution 1
class ScoreState(TypedDict):
    text: str
    score: int
    label: str

s = StateGraph(ScoreState)
s.add_node("count", lambda st: {"score": len(st["text"].split())})
s.add_edge(START, "count")
s.add_edge("count", END)
score_text = s.compile()

p = StateGraph(ScoreState)
p.add_node("score_text", score_text)
p.add_node("label", lambda st: {"label": "long" if st["score"] > 5 else "short"})
p.add_edge(START, "score_text")
p.add_edge("score_text", "label")
p.add_edge("label", END)
res = p.compile().invoke({"text": "one two three four five six seven"})
print(res)
assert res["label"] == "long"

In [ ]:
# ✅ Solution 2
def fact_checker(state: TeamState) -> Command[Literal["supervisor"]]:
    verdict = llm.invoke([SystemMessage(content="Check if the writer's answer is supported by the researcher's notes. "
                                                "Reply 'Verified' or list the problems in 1 sentence.")] + state["messages"])
    return Command(update={"messages": [AIMessage(content=verdict.content, name="fact_checker")]}, goto="supervisor")

class Next3(BaseModel):
    next: Literal["researcher", "writer", "fact_checker", "FINISH"] = Field(
        description="researcher → writer → fact_checker → FINISH")

sup3_llm = llm.with_structured_output(Next3)

def supervisor3(state: TeamState) -> Command[Literal["researcher", "writer", "fact_checker", "__end__"]]:
    steps = state.get("steps", 0)
    if steps >= 5:
        return Command(goto=END)
    history = "\n".join(f"{getattr(m, 'name', None) or m.type}: {m.content[:200]}" for m in state["messages"])
    d = sup3_llm.invoke(f"Team: researcher, writer, fact_checker.\n{history}\nWho acts next?")
    print("🧭", d.next)
    return Command(update={"steps": steps + 1}, goto=END if d.next == "FINISH" else d.next)

g = StateGraph(TeamState)
g.add_node("supervisor", supervisor3)
g.add_node("researcher", researcher)
g.add_node("writer", writer)
g.add_node("fact_checker", fact_checker)
g.add_edge(START, "supervisor")
team3 = g.compile()
out = team3.invoke({"messages": [("user", "What does Send do in LangGraph?")], "steps": 0})
print(out["messages"][-1].content)
print("✅ All exercises solved!")

## 🎯 Day 12 Recap

* A compiled graph can be a **node** (shared keys) or be **called inside a node** (different keys).
* `stream(..., subgraphs=True)` shows steps inside subgraphs; `get_graph(xray=True)` draws them.
* **Supervisor:** a router LLM picks the next specialist; specialists report back.
* **Handoffs:** agents transfer control with `Command(goto=...)`.
* Always add a **step limit** to multi-agent loops.

**Tomorrow (Day 13):** **Agentic RAG** — agents that search your own documents.